# Day 4 Project — Engineering Design Review Team
Everything is on the table: the reviewer contract, deterministic checks, a bounded fan-out, a
supervisor that reports what it merged and cut, and measurement against the golden set. Now turn
the numbers into a deployment decision and a memo.

### Step 1 — Run the whole day, both worlds

Three systems times two reviewers. With a key the live reviewer also runs once, arriving through
the same contract.

In [ ]:
SCENARIOS_TO_RUN = ("blind_spots", "strong_generalist")

def run_everything(scenario):
    model = make_mock_model(scenario)
    runs = [run_single_reviewer(SOURCE, model),
            run_checks_plus_reviewer(SOURCE, model),
            run_specialist_team(SOURCE, model)]
    return runs, [evaluate(run) for run in runs]

results = {scenario: run_everything(scenario) for scenario in SCENARIOS_TO_RUN}
for scenario, (_runs, rows) in results.items():
    print(scenario)
    for row in rows:
        print(f"   {row['system']:<24} found {row['found']}/9 | calls {row['model_calls']} | "
              f"tokens {row['tokens']} | merged {row['merged_duplicates']} | dropped {row['dropped_over_cap']}")

if LIVE:
    live_findings, live_usage = review_with_fallback(SOURCE, "general")
    live_row = score(live_findings)
    print(f"\nlive single reviewer      found {live_row['found']}/9 | 1 call | "
          f"{live_usage['total_tokens']} tokens | missed {live_row['missed']}")
else:
    print("\nNo API key: every row above came from the scripted reviewer, and the two scenarios")
    print("are the controlled A/B that makes the architecture comparison meaningful.")

### Step 2 — Read the largest system's trace end to end

Every system must be debuggable by reading: four branches, their measured usage, and exactly what
the supervisor did.

In [ ]:
team_run = results["blind_spots"][0][2]
print("System:", team_run.system, "\n")
for step in team_run.trace:
    print(step["step"])
    for key, value in step.items():
        if key != "step":
            print("   ", key, "=", value)
print("\nFinal report:")
for finding in team_run.findings:
    print(f"   {finding.severity:<8} line {finding.line:>3}  {finding.title}  [{finding.reviewer}]")

### Step 3 — Assert the bounds, measure the quality

These assertions are about *structure*: the system stayed inside its limits and every claim
carries evidence. None asserts that the team wins — that is an observation, either way.

In [ ]:
for scenario, (runs, rows) in results.items():
    for run, row in zip(runs, rows):
        assert run.model_calls <= 3, "fan-out must stay bounded"
        assert 0.0 <= row["recall"] <= 1.0
        assert row["false_positives"] == 0, "no finding may point outside the artifact"
        assert all(f.evidence.strip() for f in run.findings), "every finding needs evidence"
        assert len(run.findings) <= 20, "the supervisor must cap its report"
        assert run.raw_findings >= len(run.findings), "merging can only remove findings"
        print(f"bounds OK: {scenario}/{run.system:<22} "
              f"{run.model_calls} call(s), {len(run.findings)} findings, {run.dropped_over_cap} dropped")
print("\nStructure is asserted. Quality is measured, never asserted.")

### Step 4 — Turn a requirement into a decision

A deployment choice needs a stated quality bar. Then the rule is mechanical: **the smallest
system that clears it**.

In [ ]:
def recommend(rows, minimum_recall):
    """The smallest system (fewest calls, then fewest tokens) whose recall meets the bar."""
    qualifying = [row for row in rows if row["recall"] >= minimum_recall]
    if not qualifying:
        return None
    return min(qualifying, key=lambda row: (row["model_calls"], row["tokens"]))

BAR = 0.85
print(f"Quality bar: recall >= {BAR}\n")
for scenario, (_runs, rows) in results.items():
    choice = recommend(rows, BAR)
    print(f"{scenario:<20} " + ("no system meets the bar; do not deploy" if choice is None else
          f"deploy {choice['system']:<22} (recall {choice['recall']}, "
          f"{choice['model_calls']} call(s), {choice['tokens']} tokens)"))

### Try it yourself

Raise the bar to 1.0 — nothing may be missed. Predict each scenario's recommendation, and whether
any becomes "do not deploy".

In [ ]:
# --- Worked solution ---------------------------------------------------------------
for bar in (0.85, 1.0):
    print(f"Quality bar: recall >= {bar}")
    for scenario, (_runs, rows) in results.items():
        choice = recommend(rows, bar)
        verdict = "DO NOT DEPLOY (no system meets the bar)" if choice is None else (
            f"{choice['system']} ({choice['model_calls']} call(s), {choice['tokens']} tokens)")
        print(f"   {scenario:<20} -> {verdict}")
    print()
print("Raising the bar changes the answer, and in strong_generalist it removes every option:")
print("no amount of orchestration finds DEF-COR-02, because neither the general reviewer nor the")
print("correctness specialist can see it. The fix there is a better reviewer, a deterministic")
print("check that encodes the business rule, or a human - not another agent.")

### Step 5 — The decision memo

Built from your own numbers: a recommendation someone could disagree with by pointing at a
measurement.

In [ ]:
single_row, _augmented, team_row = results["blind_spots"][1]
strong_team = results["strong_generalist"][1][2]

print(f"""DECISION MEMO - Engineering Design Review Team

Chosen system : {team_row['system']} (world: blind_spots)
Evidence      : recall {team_row['recall']} ({team_row['found']}/9) versus {single_row['recall']}
                ({single_row['found']}/9) for a single reviewer. False positives
                {team_row['false_positives']}; duplicates surviving synthesis {team_row['duplicates']};
                the supervisor merged {team_row['merged_duplicates']} overlapping findings and
                dropped {team_row['dropped_over_cap']} over its cap.
Cost          : {team_row['model_calls']} calls / {team_row['tokens']} tokens versus
                {single_row['model_calls']} call / {single_row['tokens']} tokens - about
                {team_row['tokens'] / single_row['tokens']:.1f}x the spend for
                {team_row['found'] - single_row['found']} extra defects.
Latency       : parallel fan-out cut wall clock about 3x in section 4.4; it did not reduce
                calls or tokens.
Debugging     : 4 branches plus a merge rule instead of 1 call - more places to be wrong.
Reverses if   : the reviewer improves. Measured in the strong_generalist world, the team found
                {strong_team['found']}/9 - exactly what one reviewer found - for
                {strong_team['model_calls']} calls. Then deploy the single reviewer.""")

### Checkpoint

**1. Your team wants to add a fourth specialist (performance). What must you show before and after adding it?**

<details><summary>Show answer</summary>

The same table both ways: recall, false positives, duplicates, calls, tokens, latency, merge report. A branch is justified only if it finds defects the current system misses, at a cost you would pay knowing the number. Adding one is easy — the discipline comes from the measurement.

</details>

**2. At a bar of 1.0 no system qualifies in the `strong_generalist` world. What is the correct response?**

<details><summary>Show answer</summary>

Not "add more agents". The missed defect is invisible to the generalist *and* to the correctness specialist: a capability gap, not an attention gap. The options are a stronger reviewer, a deterministic check encoding the rule, or a human.

</details>

### Recap

- **Limitation seen:** a system can pass every structural bound and still fail the quality requirement.
- **Layer added:** a stated quality bar, a mechanical rule (smallest system that clears it), and a memo of measured numbers.
- **Evidence:** the recommendation flips between the two worlds, and flips again when the bar moves to 1.0.